# Statistics

## Sources and Interpretation

The following tables are generated during book compilation from
`milestone1/data/reported_results.json`, a transcription of Section 5 of the
updated `Hieu.docx` and `Dat.docx` reviewed on **9 November 2026**. Counts and
derived percentages are recomputed by notebook code. The original per-range run
exports are pending reconciliation; these are **document-reported aggregates**,
not results of running the crawler during book compilation.

Each document reports five ranges of 5,000 input publications. A publication is
a base arXiv ID, not a version. arXiv source outcomes and Semantic Scholar
reference outcomes measure different stages and must not be combined into a
single success percentage.

In [ ]:
from pathlib import Path
import sys

# Support execution from the project root or the notebook's own directory.
root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "myst.yml").is_file() and (p / "milestone1" / "reporting.py").is_file())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from milestone1.reporting import load_results, total, integer, percent, seconds, hms, table

data = load_results()
arxiv = data["arxiv"]["ranges"]
s2 = data["semantic_scholar"]["ranges"]

## arXiv Publication Outcomes

**Table 1.** Final reported publication statuses after Fast Crawl and Recovery.
The denominator of the `ok` rate is the assigned ID count for that row.

In [ ]:
keys = ("assigned", "metadata", "ok", "error", "no_tex", "not_found")
rows = [[f"Range {r['range']}", *[integer(r[k]) for k in keys],
         percent(r["ok"], r["assigned"])] for r in arxiv]
rows.append(["Overall", *[integer(total(arxiv, k)) for k in keys],
             percent(total(arxiv, "ok"), total(arxiv, "assigned"))])
table(["Range", "Assigned IDs", "Metadata", "ok", "error", "no_tex", "not_found", "ok / assigned"], rows)

The document reports metadata for all 25,000 IDs across 500 checkpoint batches.
There are 23,695 `ok` publications (94.78%), 22 `error` (0.088%) and 1,283
`no_tex` (5.132%), with no final `not_found` records. The `ok` rate ranges from
93.68% in Range 4 to 96.72% in Range 1, a difference of 3.04 percentage points.

These labels follow the reviewed code: a paper is `error` if any version is
`error`; otherwise it is `ok` if at least one version retained `.tex` or `.bib`,
and `no_tex` otherwise. A successful paper can still contain `no_tex` versions,
and a retained `.bib` alone can satisfy the current `ok` condition. The table
does not establish that every version has usable LaTeX.

`no_tex` accounts for about 98.31% of non-`ok` publications. It includes PDF,
empty source, no retained files and source HTTP 404/410. It cannot all be
attributed to intrinsically PDF-only submissions without examining reasons.
The 22 remaining errors can include download, format, extraction or resource
limit failures. Version counts and reason breakdowns remain pending.

## Semantic Scholar Publication Outcomes

**Table 2.** Reported statuses and cache use in the supplied rerun snapshot.
`ok` includes accepted caches, including cached empty lists. Reported
`is_api_error` counts are shown separately from historical HTTP failures.

In [ ]:
keys = ("input_papers", "ok", "not_indexed", "failed", "invalid_id", "api_errors", "zero_refs", "cache_hits")
rows = [[f"Range {r['range']}", *[integer(r[k]) for k in keys]] for r in s2]
rows.append(["Overall", *[integer(total(s2, k)) for k in keys]])
table(["Range", "Inputs", "ok incl. cache", "not_indexed", "failed", "invalid_id",
       "Reported API-error flag", "Zero refs", "Cache hits"], rows)

There are 24,980 `ok` records, 20 HTTP-404 `not_indexed` records and no reported
`failed` or `invalid_id` records in this snapshot. The aggregate status rate is
24,980 / 25,000 = **99.92%**. Of the inputs, 24,811 (99.244%) are cache hits;
only 189 are uncached valid-publication lookups. The 24,980 `ok` count therefore
does not mean 24,980 successful fresh API requests during this rerun. In this
snapshot, subtracting the reported cache hits leaves 169 fresh `ok` outcomes
and 20 `not_indexed` outcomes. A lookup may contain multiple HTTP attempts.

The zero-reference total is 1,881. These are empty records accepted by the
implementation. The current payload check can also accept a missing reference
field as empty, so the number is not independently verified absence of
citations. Likewise, zero final failure flags does not prove no HTTP 429 or
temporary network errors occurred earlier or inside retries.

## Reference Counts and arXiv Coverage

**Table 3.** All normalized local reference occurrences and the subset with
arXiv IDs. The average uses every input publication, including the 20
`not_indexed` publications whose reported reference count is zero.

In [ ]:
rows = []
for r in s2:
    rows.append([f"Range {r['range']}", integer(r["refs_total"]), integer(r["refs_with_arxiv"]),
                 integer(r["refs_total"] - r["refs_with_arxiv"]),
                 percent(r["refs_with_arxiv"], r["refs_total"]),
                 f"{r['refs_total'] / r['input_papers']:.2f}"])
all_refs, arxiv_refs, inputs = total(s2, "refs_total"), total(s2, "refs_with_arxiv"), total(s2, "input_papers")
rows.append(["Overall", integer(all_refs), integer(arxiv_refs), integer(all_refs - arxiv_refs),
             percent(arxiv_refs, all_refs), f"{all_refs / inputs:.2f}"])
table(["Range", "All local refs", "With arXiv ID", "Without arXiv ID", "arXiv share", "Refs / input paper"], rows)

The documents report **902,317** local reference occurrences. Of these,
**449,590 (49.83%)** carry arXiv IDs, while **452,727 (50.17%)** do not. The
overall average is 36.09 reference occurrences per input publication; using
only the 24,980 `ok` records as denominator gives approximately 36.12.

Range 5 has the largest local count (193,120), while Range 1 has the smallest
(163,950) and the highest arXiv-bearing share (56.21%). Keeping references
without arXiv IDs locally preserves a substantial part of the collected
metadata, even though the published reference pool uses the arXiv-bearing
subset.

These counts are citation occurrences associated with citing publications,
not the number of globally unique cited works. The same work cited by two
different papers contributes two relations. Counts before and after the
identifier-tuple deduplication, and global distinct-work counts, are not
established by the supplied aggregates.

## Reported Table Validation

**Table 4.** Output row counts and findings of the current validator, as
reported in Dat's document. Tables are exported even when issues are present.

In [ ]:
rows = [[f"Range {r['range']}", integer(r["input_papers"]), integer(r["validation_references"]),
         integer(r["validation_issues"]), "No issues reported" if not r["validation_issues"] else "Review pending"] for r in s2]
rows.append(["Overall", integer(total(s2, "input_papers")), integer(total(s2, "validation_references")),
             integer(total(s2, "validation_issues")), "Two metadata issues pending"])
table(["Range", "papers rows", "references rows", "Validator issues", "Review status"], rows)
details = data["semantic_scholar"]["validation_details"]
table(["Range", "Table", "Reported index", "Field", "Finding"],
      [[f"Range {r['range']}", r["table"], str(r["reported_index"]), r["field"], r["problem"]] for r in details])

Ranges 1--3 report no validator issues. Ranges 4 and 5 each report an empty
`ref_authors` list, at the indices shown above. These are findings under the
implemented non-empty-author validation rule; their source records and the
agreed missing-metadata policy are pending review. No missing authors are
invented and no records are removed merely to force a passing result.

The validator checks selected columns, types, emptiness and duplicate keys.
It does not establish complete bibliographic accuracy, all date formats or all
cross-table relationships. A cited `ref_arxiv_id` is allowed to be outside the
assigned `papers` corpus.

## Remaining Evidence

Reconciliation requires the original per-range `summary.json`, `metrics.csv`,
`runtime_history.csv`, `crawl_records.csv` and `failed_versions.csv` for arXiv,
and `metrics_s2_references.json`, `s2_crawl_records.csv`,
`validation_report.json` for Semantic Scholar. Version coverage, error/no-TeX
reasons and the two author issues can then be checked against records.

The reported figures above remain usable as attributed current results while
those exports and the requested corrections are pending. Neither a fresh crawl
nor any external API request is performed by this notebook.